# Experiment — FEVER
Main sweep: 3 models × 3 prompt types × {r=0, r=1} on FEVER (N=50).
Results saved to `results/fever_results.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import yaml
import pandas as pd
from pathlib import Path

with open("../configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

MODELS       = cfg["models"]["available"]
PROMPT_TYPES = ["standard", "chain_of_thought", "vigilant"]
N            = cfg["evaluation"]["n_examples"]
SEED         = cfg["seed"]
RESULTS_DIR  = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Models: {MODELS}")
print(f"Prompt types: {PROMPT_TYPES}")
print(f"N: {N}  seed: {SEED}  k: {cfg['retrieval']['k']}")

In [ ]:
from src.data.fever_loader import load_fever

POOL_SIZE = 200  # fixed pool so cache stays valid when N changes
all_clean = load_fever("../" + cfg["dataset"]["fever_dev"], max_examples=POOL_SIZE)
examples = all_clean[:N]  # same Python objects — identity-based exclusion works
print(f"Pool: {len(all_clean)}  Evaluated: {len(examples)}")
print("Label distribution:", {l: sum(1 for e in examples if e["label"] == l)
                              for l in {e["label"] for e in examples}})

In [ ]:
from src.retrieval.embedder import Embedder
from src.retrieval.retriever import Retriever

emb_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["embeddings_subdir"])
embedder = Embedder(model_name=cfg["retrieval"]["embedding_model"], cache_dir=emb_cache)
retriever = Retriever(embedder=embedder, k=cfg["retrieval"]["k"])
print(f"Embedder: {cfg['retrieval']['embedding_model']}  k={cfg['retrieval']['k']}")

In [ ]:
from src.data.fever_loader import load_fever
from src.evaluation.scorer import run as run_scorer
from src.generation.llm_client import HuggingFaceClient

all_poisoned = load_fever("../" + cfg["dataset"]["fever_dev_poisoned"])
examples_poisoned = all_poisoned[:N]  # same objects — identity exclusion works
print(f"Clean examples   : {len(examples)}")
print(f"Poisoned examples: {len(examples_poisoned)}")

records = []

for model in MODELS:
    print(f"\n{'='*60}\nModel: {model}\n{'='*60}")
    llm_cache = os.path.join("../", cfg["cache"]["dir"], cfg["cache"]["llm_subdir"])
    llm = HuggingFaceClient(
        model=model,
        temperature=cfg["models"]["temperature"],
        cache_dir=llm_cache,
    )
    with embedder, llm:
        for prompt_type in PROMPT_TYPES:
            print(f"  prompt={prompt_type}  r=0", end="")
            m0 = run_scorer(
                examples=examples, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_clean,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 0, **m0})
            print(f" acc={m0['accuracy']:.3f}  r=1", end="")
            m1 = run_scorer(
                examples=examples_poisoned, retriever=retriever, llm=llm,
                prompt_type=prompt_type, seed=SEED,
                full_dataset=all_poisoned,
            )
            records.append({"model": model, "prompt_type": prompt_type,
                            "poison_rate": 1, **m1})
            print(f" acc={m1['accuracy']:.3f}")

print("\nSweep complete.")

In [ ]:
df = pd.DataFrame(records)
out_path = RESULTS_DIR / "fever_results.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} rows to {out_path}")
df